# Task 2: Create a Prescription Parser using CRF
This task tests your ability to build a Doctor Prescription Parser with the help of CRF model

Your job is to build a Prescription Parser that takes a prescription (sentence) as an input and find / label the words in that sentence with one of the already pre-defined labels

### Problem: SEQUENCE PREDICTION - Label words in a sentence
#### Input : Doctor Prescription in the form of a sentence split into tokens
- Ex: Take 2 tablets once a day for 10 days

#### Output : FHIR Labels
- ('Take', 'Method')
- ('2', 'Qty') 
- ('tablets', 'Form')
- ('once', 'Frequency')
- ('a', 'Period') 
- ('day', 'PeriodUnit')
- ('for', 'FOR')
- ('10', 'Duration')
- ('days', 'DurationUnit') 

### Major Steps
- Install necessary library
- Import the libraries
- Create training data with labels
    - Split the sentence into tokens
    - Compute POS tags
    - Create triples
- Extract features
- Split the data into training and testing set
- Create CRF model
- Save the CRF model
- Load the CRF model
- Predict on test data
- Accuracy

#### Install necesaary library

#### Import the necessary libraries

In [7]:
import nltk
import sklearn
import sklearn_crfsuite
from sklearn_crfsuite import CRF
from sklearn_crfsuite.metrics import flat_classification_report
from nltk.tokenize import word_tokenize
from nltk import pos_tag
import random
import joblib

### Input data (GIVEN)
#### Creating the inputs to the ML model in the following form:
- sigs --> ['take 3 tabs for 10 days']       INPUT SIG
- input_sigs --> [['take', '3', 'tabs', 'for', '10', 'days']]      TOKENS
- output_labels --> [['Method','Qty', 'Form', 'FOR', 'Duration', 'DurationUnit']]       LABELS

In [9]:
sigs = ["for 5 to 6 days", "inject 2 units", "x 2 weeks", "x 3 days", "every day", "every 2 weeks", "every 3 days", "every 1 to 2 months", "every 2 to 6 weeks", "every 4 to 6 days", "take two to four tabs", "take 2 to 4 tabs", "take 3 tabs orally bid for 10 days at bedtime", "swallow three capsules tid orally", "take 2 capsules po every 6 hours", "take 2 tabs po for 10 days", "take 100 caps by mouth tid for 10 weeks", "take 2 tabs after an hour", "2 tabs every 4-6 hours", "every 4 to 6 hours", "q46h", "q4-6h", "2 hours before breakfast", "before 30 mins at bedtime", "30 mins before bed", "and 100 tabs twice a month", "100 tabs twice a month", "100 tabs once a month", "100 tabs thrice a month", "3 tabs daily for 3 days then 1 tab per day at bed", "30 tabs 10 days tid", "take 30 tabs for 10 days three times a day", "qid q6h", "bid", "qid", "30 tabs before dinner and bedtime", "30 tabs before dinner & bedtime", "take 3 tabs at bedtime", "30 tabs thrice daily for 10 days ", "30 tabs for 10 days three times a day", "Take 2 tablets a day", "qid for 10 days", "every day", "take 2 caps at bedtime", "apply 3 drops before bedtime", "take three capsules daily", "swallow 3 pills once a day", "swallow three pills thrice a day", "apply daily", "apply three drops before bedtime", "every 6 hours", "before food", "after food", "for 20 days", "for twenty days", "with meals"]
input_sigs = [['for', '5', 'to', '6', 'days'], ['inject', '2', 'units'], ['x', '2', 'weeks'], ['x', '3', 'days'], ['every', 'day'], ['every', '2', 'weeks'], ['every', '3', 'days'], ['every', '1', 'to', '2', 'months'], ['every', '2', 'to', '6', 'weeks'], ['every', '4', 'to', '6', 'days'], ['take', 'two', 'to', 'four', 'tabs'], ['take', '2', 'to', '4', 'tabs'], ['take', '3', 'tabs', 'orally', 'bid', 'for', '10', 'days', 'at', 'bedtime'], ['swallow', 'three', 'capsules', 'tid', 'orally'], ['take', '2', 'capsules', 'po', 'every', '6', 'hours'], ['take', '2', 'tabs', 'po', 'for', '10', 'days'], ['take', '100', 'caps', 'by', 'mouth', 'tid', 'for', '10', 'weeks'], ['take', '2', 'tabs', 'after', 'an', 'hour'], ['2', 'tabs', 'every', '4-6', 'hours'], ['every', '4', 'to', '6', 'hours'], ['q46h'], ['q4-6h'], ['2', 'hours', 'before', 'breakfast'], ['before', '30', 'mins', 'at', 'bedtime'], ['30', 'mins', 'before', 'bed'], ['and', '100', 'tabs', 'twice', 'a', 'month'], ['100', 'tabs', 'twice', 'a', 'month'], ['100', 'tabs', 'once', 'a', 'month'], ['100', 'tabs', 'thrice', 'a', 'month'], ['3', 'tabs', 'daily', 'for', '3', 'days', 'then', '1', 'tab', 'per', 'day', 'at', 'bed'], ['30', 'tabs', '10', 'days', 'tid'], ['take', '30', 'tabs', 'for', '10', 'days', 'three', 'times', 'a', 'day'], ['qid', 'q6h'], ['bid'], ['qid'], ['30', 'tabs', 'before', 'dinner', 'and', 'bedtime'], ['30', 'tabs', 'before', 'dinner', '&', 'bedtime'], ['take', '3', 'tabs', 'at', 'bedtime'], ['30', 'tabs', 'thrice', 'daily', 'for', '10', 'days'], ['30', 'tabs', 'for', '10', 'days', 'three', 'times', 'a', 'day'], ['take', '2', 'tablets', 'a', 'day'], ['qid', 'for', '10', 'days'], ['every', 'day'], ['take', '2', 'caps', 'at', 'bedtime'], ['apply', '3', 'drops', 'before', 'bedtime'], ['take', 'three', 'capsules', 'daily'], ['swallow', '3', 'pills', 'once', 'a', 'day'], ['swallow', 'three', 'pills', 'thrice', 'a', 'day'], ['apply', 'daily'], ['apply', 'three', 'drops', 'before', 'bedtime'], ['every', '6', 'hours'], ['before', 'food'], ['after', 'food'], ['for', '20', 'days'], ['for', 'twenty', 'days'], ['with', 'meals']]
output_labels = [['FOR', 'Duration', 'TO', 'DurationMax', 'DurationUnit'], ['Method', 'Qty', 'Form'], ['FOR', 'Duration', 'DurationUnit'], ['FOR', 'Duration', 'DurationUnit'], ['EVERY', 'Period'], ['EVERY', 'Period', 'PeriodUnit'], ['EVERY', 'Period', 'PeriodUnit'], ['EVERY', 'Period', 'TO', 'PeriodMax', 'PeriodUnit'], ['EVERY', 'Period', 'TO', 'PeriodMax', 'PeriodUnit'], ['EVERY', 'Period', 'TO', 'PeriodMax', 'PeriodUnit'], ['Method', 'Qty', 'TO', 'Qty', 'Form'], ['Method', 'Qty', 'TO', 'Qty', 'Form'], ['Method', 'Qty', 'Form', 'PO', 'BID', 'FOR', 'Duration', 'DurationUnit', 'AT', 'WHEN'], ['Method', 'Qty', 'Form', 'TID', 'PO'], ['Method', 'Qty', 'Form', 'PO', 'EVERY', 'Period', 'PeriodUnit'], ['Method', 'Qty', 'Form', 'PO', 'FOR', 'Duration', 'DurationUnit'], ['Method', 'Qty', 'Form', 'BY', 'PO', 'TID', 'FOR', 'Duration', 'DurationUnit'], ['Method', 'Qty', 'Form', 'AFTER', 'Period', 'PeriodUnit'], ['Qty', 'Form', 'EVERY', 'Period', 'PeriodUnit'], ['EVERY', 'Period', 'TO', 'PeriodMax', 'PeriodUnit'], ['Q46H'], ['Q4-6H'], ['Qty', 'PeriodUnit', 'BEFORE', 'WHEN'], ['BEFORE', 'Qty', 'M', 'AT', 'WHEN'], ['Qty', 'M', 'BEFORE', 'WHEN'], ['AND', 'Qty', 'Form', 'Frequency', 'Period', 'PeriodUnit'], ['Qty', 'Form', 'Frequency', 'Period', 'PeriodUnit'], ['Qty', 'Form', 'Frequency', 'Period', 'PeriodUnit'], ['Qty', 'Form', 'Frequency', 'Period', 'PeriodUnit'], ['Qty', 'Form', 'Frequency', 'FOR', 'Duration', 'DurationUnit', 'THEN', 'Qty', 'Form', 'Frequency', 'PeriodUnit', 'AT', 'WHEN'], ['Qty', 'Form', 'Duration', 'DurationUnit', 'TID'], ['Method', 'Qty', 'Form', 'FOR', 'Duration', 'DurationUnit', 'Qty', 'TIMES', 'Period', 'PeriodUnit'], ['QID', 'Q6H'], ['BID'], ['QID'],['Qty', 'Form', 'BEFORE', 'WHEN', 'AND', 'WHEN'], ['Qty', 'Form', 'BEFORE', 'WHEN', 'AND', 'WHEN'], ['Method', 'Qty', 'Form', 'AT', 'WHEN'], ['Qty', 'Form', 'Frequency', 'DAILY', 'FOR', 'Duration', 'DurationUnit'], ['Qty', 'Form', 'FOR', 'Duration', 'DurationUnit', 'Frequency', 'TIMES', 'Period', 'PeriodUnit'], ['Method', 'Qty', 'Form', 'Period', 'PeriodUnit'], ['QID', 'FOR', 'Duration', 'DurationUnit'], ['EVERY', 'PeriodUnit'], ['Method', 'Qty', 'Form', 'AT', 'WHEN'], ['Method', 'Qty', 'Form', 'BEFORE', 'WHEN'], ['Method', 'Qty', 'Form', 'DAILY'], ['Method', 'Qty', 'Form', 'Frequency', 'Period', 'PeriodUnit'], ['Method', 'Qty', 'Form', 'Frequency', 'Period', 'PeriodUnit'], ['Method', 'DAILY'], ['Method', 'Qty', 'Form', 'BEFORE', 'WHEN'], ['EVERY', 'Period', 'PeriodUnit'], ['BEFORE', 'FOOD'], ['AFTER', 'FOOD'], ['FOR', 'Duration', 'DurationUnit'], ['FOR', 'Duration', 'DurationUnit'], ['WITH', 'FOOD']]

In [11]:
len(sigs), len(input_sigs) , len(output_labels)

(56, 56, 56)

### Creating a Tuples Maker method
Create the tuples as given below by writing a function **tuples_maker(input_sigs, output_labels)** and returns **output** as given below

Input(s): 
- input_sigs
- output_lables

Output:

[[('for', 'FOR'),
  ('5', 'Duration'),
  ('to', 'TO'),
  ('6', 'DurationMax'),
  ('days', 'DurationUnit')], [second sentence], ...]

In [23]:
def tuples_maker(inp, out):
    
     return [list(zip(tokens, labels)) for tokens, labels in zip(inp, out)]

In [37]:
tuples_maker_output = tuples_maker(input_sigs, output_labels)
print(tuples_maker_output)

[[('for', 'FOR'), ('5', 'Duration'), ('to', 'TO'), ('6', 'DurationMax'), ('days', 'DurationUnit')], [('inject', 'Method'), ('2', 'Qty'), ('units', 'Form')], [('x', 'FOR'), ('2', 'Duration'), ('weeks', 'DurationUnit')], [('x', 'FOR'), ('3', 'Duration'), ('days', 'DurationUnit')], [('every', 'EVERY'), ('day', 'Period')], [('every', 'EVERY'), ('2', 'Period'), ('weeks', 'PeriodUnit')], [('every', 'EVERY'), ('3', 'Period'), ('days', 'PeriodUnit')], [('every', 'EVERY'), ('1', 'Period'), ('to', 'TO'), ('2', 'PeriodMax'), ('months', 'PeriodUnit')], [('every', 'EVERY'), ('2', 'Period'), ('to', 'TO'), ('6', 'PeriodMax'), ('weeks', 'PeriodUnit')], [('every', 'EVERY'), ('4', 'Period'), ('to', 'TO'), ('6', 'PeriodMax'), ('days', 'PeriodUnit')], [('take', 'Method'), ('two', 'Qty'), ('to', 'TO'), ('four', 'Qty'), ('tabs', 'Form')], [('take', 'Method'), ('2', 'Qty'), ('to', 'TO'), ('4', 'Qty'), ('tabs', 'Form')], [('take', 'Method'), ('3', 'Qty'), ('tabs', 'Form'), ('orally', 'PO'), ('bid', 'BID'), ('

### Creating the triples_maker( ) for feature extraction
- input: tuples_maker_output
- output: 
[[('for', 'IN', 'FOR'),
  ('5', 'CD', 'Duration'),
  ('to', 'TO', 'TO'),
  ('6', 'CD', 'DurationMax'),
  ('days', 'NNS', 'DurationUnit')], [second sentence], ... ]

### Creating the features extractor method (GIVEN as a BASELINE)
#### The features used are:
- SOS, EOS, lowercase, uppercase, title, digit, postag, previous_tag, next_tag
#### Feel free to include more features
### Running the feature extractor on the training data
- Feature extraction
- Train-test-split

In [289]:
import nltk
import sklearn_crfsuite
from sklearn.model_selection import train_test_split
from sklearn_crfsuite import metrics

# ---------------------------
# 1. Provided Data
# ---------------------------
sigs = [
    "for 5 to 6 days", "inject 2 units", "x 2 weeks", "x 3 days", "every day", 
    "every 2 weeks", "every 3 days", "every 1 to 2 months", "every 2 to 6 weeks", 
    "every 4 to 6 days", "take two to four tabs", "take 2 to 4 tabs", 
    "take 3 tabs orally bid for 10 days at bedtime", "swallow three capsules tid orally", 
    "take 2 capsules po every 6 hours", "take 2 tabs po for 10 days", 
    "take 100 caps by mouth tid for 10 weeks", "take 2 tabs after an hour", 
    "2 tabs every 4-6 hours", "every 4 to 6 hours", "q46h", "q4-6h", 
    "2 hours before breakfast", "before 30 mins at bedtime", "30 mins before bed", 
    "and 100 tabs twice a month", "100 tabs twice a month", "100 tabs once a month", 
    "100 tabs thrice a month", "3 tabs daily for 3 days then 1 tab per day at bed", 
    "30 tabs 10 days tid", "take 30 tabs for 10 days three times a day", "qid q6h", 
    "bid", "qid", "30 tabs before dinner and bedtime", "30 tabs before dinner & bedtime", 
    "take 3 tabs at bedtime", "30 tabs thrice daily for 10 days ", 
    "30 tabs for 10 days three times a day", "Take 2 tablets a day", "qid for 10 days", 
    "every day", "take 2 caps at bedtime", "apply 3 drops before bedtime", 
    "take three capsules daily", "swallow 3 pills once a day", "swallow three pills thrice a day", 
    "apply daily", "apply three drops before bedtime", "every 6 hours", "before food", 
    "after food", "for 20 days", "for twenty days", "with meals"
]

input_sigs = [
    ['for', '5', 'to', '6', 'days'], ['inject', '2', 'units'], ['x', '2', 'weeks'], 
    ['x', '3', 'days'], ['every', 'day'], ['every', '2', 'weeks'], ['every', '3', 'days'], 
    ['every', '1', 'to', '2', 'months'], ['every', '2', 'to', '6', 'weeks'], 
    ['every', '4', 'to', '6', 'days'], ['take', 'two', 'to', 'four', 'tabs'], 
    ['take', '2', 'to', '4', 'tabs'], ['take', '3', 'tabs', 'orally', 'bid', 'for', '10', 'days', 'at', 'bedtime'], 
    ['swallow', 'three', 'capsules', 'tid', 'orally'], ['take', '2', 'capsules', 'po', 'every', '6', 'hours'], 
    ['take', '2', 'tabs', 'po', 'for', '10', 'days'], ['take', '100', 'caps', 'by', 'mouth', 'tid', 'for', '10', 'weeks'], 
    ['take', '2', 'tabs', 'after', 'an', 'hour'], ['2', 'tabs', 'every', '4-6', 'hours'], 
    ['every', '4', 'to', '6', 'hours'], ['q46h'], ['q4-6h'], ['2', 'hours', 'before', 'breakfast'], 
    ['before', '30', 'mins', 'at', 'bedtime'], ['30', 'mins', 'before', 'bed'], 
    ['and', '100', 'tabs', 'twice', 'a', 'month'], ['100', 'tabs', 'twice', 'a', 'month'], 
    ['100', 'tabs', 'once', 'a', 'month'], ['100', 'tabs', 'thrice', 'a', 'month'], 
    ['3', 'tabs', 'daily', 'for', '3', 'days', 'then', '1', 'tab', 'per', 'day', 'at', 'bed'], 
    ['30', 'tabs', '10', 'days', 'tid'], ['take', '30', 'tabs', 'for', '10', 'days', 'three', 'times', 'a', 'day'], 
    ['qid', 'q6h'], ['bid'], ['qid'], ['30', 'tabs', 'before', 'dinner', 'and', 'bedtime'], 
    ['30', 'tabs', 'before', 'dinner', '&', 'bedtime'], ['take', '3', 'tabs', 'at', 'bedtime'], 
    ['30', 'tabs', 'thrice', 'daily', 'for', '10', 'days'], ['30', 'tabs', 'for', '10', 'days', 'three', 'times', 'a', 'day'], 
    ['take', '2', 'tablets', 'a', 'day'], ['qid', 'for', '10', 'days'], ['every', 'day'], 
    ['take', '2', 'caps', 'at', 'bedtime'], ['apply', '3', 'drops', 'before', 'bedtime'], 
    ['take', 'three', 'capsules', 'daily'], ['swallow', '3', 'pills', 'once', 'a', 'day'], 
    ['swallow', 'three', 'pills', 'thrice', 'a', 'day'], ['apply', 'daily'], 
    ['apply', 'three', 'drops', 'before', 'bedtime'], ['every', '6', 'hours'], ['before', 'food'], 
    ['after', 'food'], ['for', '20', 'days'], ['for', 'twenty', 'days'], ['with', 'meals']
]

output_labels = [
    ['FOR', 'Duration', 'TO', 'DurationMax', 'DurationUnit'], ['Method', 'Qty', 'Form'], 
    ['FOR', 'Duration', 'DurationUnit'], ['FOR', 'Duration', 'DurationUnit'], ['EVERY', 'Period'], 
    ['EVERY', 'Period', 'PeriodUnit'], ['EVERY', 'Period', 'PeriodUnit'], 
    ['EVERY', 'Period', 'TO', 'PeriodMax', 'PeriodUnit'], ['EVERY', 'Period', 'TO', 'PeriodMax', 'PeriodUnit'], 
    ['EVERY', 'Period', 'TO', 'PeriodMax', 'PeriodUnit'], ['Method', 'Qty', 'TO', 'Qty', 'Form'], 
    ['Method', 'Qty', 'TO', 'Qty', 'Form'], ['Method', 'Qty', 'Form', 'PO', 'BID', 'FOR', 'Duration', 'DurationUnit', 'AT', 'WHEN'], 
    ['Method', 'Qty', 'Form', 'TID', 'PO'], ['Method', 'Qty', 'Form', 'PO', 'EVERY', 'Period', 'PeriodUnit'], 
    ['Method', 'Qty', 'Form', 'PO', 'FOR', 'Duration', 'DurationUnit'], 
    ['Method', 'Qty', 'Form', 'BY', 'PO', 'TID', 'FOR', 'Duration', 'DurationUnit'], 
    ['Method', 'Qty', 'Form', 'AFTER', 'Period', 'PeriodUnit'], ['Qty', 'Form', 'EVERY', 'Period', 'PeriodUnit'], 
    ['EVERY', 'Period', 'TO', 'PeriodMax', 'PeriodUnit'], ['Q46H'], ['Q4-6H'], 
    ['Qty', 'PeriodUnit', 'BEFORE', 'WHEN'], ['BEFORE', 'Qty', 'M', 'AT', 'WHEN'], 
    ['Qty', 'M', 'BEFORE', 'WHEN'], ['AND', 'Qty', 'Form', 'Frequency', 'Period', 'PeriodUnit'], 
    ['Qty', 'Form', 'Frequency', 'Period', 'PeriodUnit'], ['Qty', 'Form', 'Frequency', 'Period', 'PeriodUnit'], 
    ['Qty', 'Form', 'Frequency', 'Period', 'PeriodUnit'], 
    ['Qty', 'Form', 'Frequency', 'FOR', 'Duration', 'DurationUnit', 'THEN', 'Qty', 'Form', 'Frequency', 'PeriodUnit', 'AT', 'WHEN'], 
    ['Qty', 'Form', 'Duration', 'DurationUnit', 'TID'], 
    ['Method', 'Qty', 'Form', 'FOR', 'Duration', 'DurationUnit', 'Qty', 'TIMES', 'Period', 'PeriodUnit'], 
    ['QID', 'Q6H'], ['BID'], ['QID'], 
    ['Qty', 'Form', 'BEFORE', 'WHEN', 'AND', 'WHEN'], ['Qty', 'Form', 'BEFORE', 'WHEN', 'AND', 'WHEN'], 
    ['Method', 'Qty', 'Form', 'AT', 'WHEN'], ['Qty', 'Form', 'Frequency', 'DAILY', 'FOR', 'Duration', 'DurationUnit'], 
    ['Qty', 'Form', 'FOR', 'Duration', 'DurationUnit', 'Frequency', 'TIMES', 'Period', 'PeriodUnit'], 
    ['Method', 'Qty', 'Form', 'Period', 'PeriodUnit'], ['QID', 'FOR', 'Duration', 'DurationUnit'], 
    ['EVERY', 'PeriodUnit'], ['Method', 'Qty', 'Form', 'AT', 'WHEN'], ['Method', 'Qty', 'Form', 'BEFORE', 'WHEN'], 
    ['Method', 'Qty', 'Form', 'DAILY'], ['Method', 'Qty', 'Form', 'Frequency', 'Period', 'PeriodUnit'], 
    ['Method', 'Qty', 'Form', 'Frequency', 'Period', 'PeriodUnit'], ['Method', 'DAILY'], 
    ['Method', 'Qty', 'Form', 'BEFORE', 'WHEN'], ['EVERY', 'Period', 'PeriodUnit'], 
    ['BEFORE', 'FOOD'], ['AFTER', 'FOOD'], ['FOR', 'Duration', 'DurationUnit'], 
    ['FOR', 'Duration', 'DurationUnit'], ['WITH', 'FOOD']
]

data = []  # List to hold sentences (each sentence is a list of (token, pos, label))
for tokens, labels in zip(input_sigs, output_labels):
    pos_tags = nltk.pos_tag(tokens) #Averaged Perceptron Tagger
    # Optional: Check if the number of tokens and labels match
    if len(tokens) != len(labels):
        print("Warning: Mismatch in token and label count for sentence:", tokens)
    sentence_data = []
    for (token, pos), label in zip(pos_tags, labels):
        sentence_data.append((token, pos, label))
    data.append(sentence_data)

In [272]:
def word2features(sent, i):
    """
    Extract features for a given token in a sentence.
    Each token is represented by a dictionary of features.
    """
    token = sent[i][0]
    pos = sent[i][1]
    features = {
        'bias': 1.0,
        'word.lower()': token.lower(),
        'word[-3:]': token[-3:],
        'word[-2:]': token[-2:],
        'word.isupper()': token.isupper(),
        'word.istitle()': token.istitle(),
        'word.isdigit()': token.isdigit(),
        'pos': pos,
    }
    if i > 0:
        token_prev = sent[i-1][0]
        pos_prev = sent[i-1][1]
        features.update({
            '-1:word.lower()': token_prev.lower(),
            '-1:pos': pos_prev,
        })
    else:
        features['BOS'] = True  # Beginning of Sentence
    if i < len(sent) - 1:
        token_next = sent[i+1][0]
        pos_next = sent[i+1][1]
        features.update({
            '+1:word.lower()': token_next.lower(),
            '+1:pos': pos_next,
        })
    else:
        features['EOS'] = True  # End of Sentence
    return features

def sent2features(sent):
    """Extract features for all tokens in a sentence."""
    return [word2features(sent, i) for i in range(len(sent))]

def sent2labels(sent):
    """Extract labels for all tokens in a sentence."""
    return [label for token, pos, label in sent]

def sent2tokens(sent):
    """Extract tokens from a sentence (for reference or debugging)."""
    return [token for token, pos, label in sent]

# Prepare the feature set (X) and corresponding labels (y)
X = [sent2features(s) for s in data]
y = [sent2labels(s) for s in data]

In [282]:
import pprint
pprint.pprint(data)

[[('for', 'IN', 'FOR'),
  ('5', 'CD', 'Duration'),
  ('to', 'TO', 'TO'),
  ('6', 'CD', 'DurationMax'),
  ('days', 'NNS', 'DurationUnit')],
 [('inject', 'JJ', 'Method'), ('2', 'CD', 'Qty'), ('units', 'NNS', 'Form')],
 [('x', 'RB', 'FOR'),
  ('2', 'CD', 'Duration'),
  ('weeks', 'NNS', 'DurationUnit')],
 [('x', 'RB', 'FOR'), ('3', 'CD', 'Duration'), ('days', 'NNS', 'DurationUnit')],
 [('every', 'DT', 'EVERY'), ('day', 'NN', 'Period')],
 [('every', 'DT', 'EVERY'),
  ('2', 'CD', 'Period'),
  ('weeks', 'NNS', 'PeriodUnit')],
 [('every', 'DT', 'EVERY'),
  ('3', 'CD', 'Period'),
  ('days', 'NNS', 'PeriodUnit')],
 [('every', 'DT', 'EVERY'),
  ('1', 'CD', 'Period'),
  ('to', 'TO', 'TO'),
  ('2', 'CD', 'PeriodMax'),
  ('months', 'NNS', 'PeriodUnit')],
 [('every', 'DT', 'EVERY'),
  ('2', 'CD', 'Period'),
  ('to', 'TO', 'TO'),
  ('6', 'CD', 'PeriodMax'),
  ('weeks', 'NNS', 'PeriodUnit')],
 [('every', 'DT', 'EVERY'),
  ('4', 'CD', 'Period'),
  ('to', 'TO', 'TO'),
  ('6', 'CD', 'PeriodMax'),
  ('days

In [295]:
import pprint
pprint.pprint(X)

[[{'+1:pos': 'CD',
   '+1:word.lower()': '5',
   'BOS': True,
   'bias': 1.0,
   'pos': 'IN',
   'word.isdigit()': False,
   'word.istitle()': False,
   'word.isupper()': False,
   'word.lower()': 'for',
   'word[-2:]': 'or',
   'word[-3:]': 'for'},
  {'+1:pos': 'TO',
   '+1:word.lower()': 'to',
   '-1:pos': 'IN',
   '-1:word.lower()': 'for',
   'bias': 1.0,
   'pos': 'CD',
   'word.isdigit()': True,
   'word.istitle()': False,
   'word.isupper()': False,
   'word.lower()': '5',
   'word[-2:]': '5',
   'word[-3:]': '5'},
  {'+1:pos': 'CD',
   '+1:word.lower()': '6',
   '-1:pos': 'CD',
   '-1:word.lower()': '5',
   'bias': 1.0,
   'pos': 'TO',
   'word.isdigit()': False,
   'word.istitle()': False,
   'word.isupper()': False,
   'word.lower()': 'to',
   'word[-2:]': 'to',
   'word[-3:]': 'to'},
  {'+1:pos': 'NNS',
   '+1:word.lower()': 'days',
   '-1:pos': 'TO',
   '-1:word.lower()': 'to',
   'bias': 1.0,
   'pos': 'CD',
   'word.isdigit()': True,
   'word.istitle()': False,
   'word.is

In [297]:
import pprint
pprint.pprint(y)

[['FOR', 'Duration', 'TO', 'DurationMax', 'DurationUnit'],
 ['Method', 'Qty', 'Form'],
 ['FOR', 'Duration', 'DurationUnit'],
 ['FOR', 'Duration', 'DurationUnit'],
 ['EVERY', 'Period'],
 ['EVERY', 'Period', 'PeriodUnit'],
 ['EVERY', 'Period', 'PeriodUnit'],
 ['EVERY', 'Period', 'TO', 'PeriodMax', 'PeriodUnit'],
 ['EVERY', 'Period', 'TO', 'PeriodMax', 'PeriodUnit'],
 ['EVERY', 'Period', 'TO', 'PeriodMax', 'PeriodUnit'],
 ['Method', 'Qty', 'TO', 'Qty', 'Form'],
 ['Method', 'Qty', 'TO', 'Qty', 'Form'],
 ['Method',
  'Qty',
  'Form',
  'PO',
  'BID',
  'FOR',
  'Duration',
  'DurationUnit',
  'AT',
  'WHEN'],
 ['Method', 'Qty', 'Form', 'TID', 'PO'],
 ['Method', 'Qty', 'Form', 'PO', 'EVERY', 'Period', 'PeriodUnit'],
 ['Method', 'Qty', 'Form', 'PO', 'FOR', 'Duration', 'DurationUnit'],
 ['Method',
  'Qty',
  'Form',
  'BY',
  'PO',
  'TID',
  'FOR',
  'Duration',
  'DurationUnit'],
 ['Method', 'Qty', 'Form', 'AFTER', 'Period', 'PeriodUnit'],
 ['Qty', 'Form', 'EVERY', 'Period', 'PeriodUnit'],
 

In [66]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

### Training the CRF model with the features extracted using the feature extractor method

In [68]:


# Submit training data to the trainer
# Set the parameters of the model
crf = sklearn_crfsuite.CRF(
    algorithm='lbfgs',
    max_iterations=100,
    all_possible_transitions=True
)

crf.fit(X_train, y_train)

# Providing a file name as a parameter to the train function, such that
# the model will be saved to the file when training is finished


CRF(algorithm='lbfgs', all_possible_transitions=True, max_iterations=100)

### Predicting the test data with the built model

In [72]:
y_pred = crf.predict(X_test)
report = metrics.flat_classification_report(y_test, y_pred, digits=3)
print("CRF Model Performance:\n")
print(report)

CRF Model Performance:

              precision    recall  f1-score   support

         AND      1.000     1.000     1.000         1
          AT      1.000     1.000     1.000         1
      BEFORE      1.000     1.000     1.000         2
         BID      0.000     0.000     0.000         2
       DAILY      0.000     0.000     0.000         0
    Duration      1.000     1.000     1.000         4
 DurationMax      0.000     0.000     0.000         1
DurationUnit      1.000     0.750     0.857         4
       EVERY      1.000     1.000     1.000         3
         FOR      1.000     1.000     1.000         4
        Form      1.000     1.000     1.000         5
   Frequency      0.333     1.000     0.500         1
      Method      1.000     1.000     1.000         3
          PO      0.000     0.000     0.000         2
      Period      1.000     1.000     1.000         4
   PeriodMax      0.500     1.000     0.667         1
  PeriodUnit      0.667     1.000     0.800         4
   

C:\Users\wayne\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\wayne\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\wayne\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\wayne\anaconda3\Lib\site-packages\sklea

### Putting all the prediction logic inside a predict method

In [180]:
def predict(sentence):
    # 简单的空格分词
    tokens = sentence.split()
    # 使用 NLTK 进行 POS 标注
    pos_tags = nltk.pos_tag(tokens)
    # 构造句子数据（标签位置可以使用 None，因为预测时不需要标签）
    sent = [(token, pos, None) for token, pos in pos_tags]
    # 提取特征
    features = sent2features(sent)
    # 注意：crf.predict 需要传入包含多个句子的列表，因此这里用 [features]
    predicted_labels = crf.predict([features])
    return tokens, predicted_labels[0]

### Sample predictions

In [182]:
predictions = predict("take 2 tabs every 6 hours x 10 days")
print (predictions)

(['take', '2', 'tabs', 'every', '6', 'hours', 'x', '10', 'days'], array(['Method', 'Qty', 'Form', 'EVERY', 'Period', 'PeriodUnit', 'FOR',
       'Duration', 'DurationUnit'], dtype=object))


In [184]:
predictions = predict("2 capsu for 10 day at bed")
print (predictions)

(['2', 'capsu', 'for', '10', 'day', 'at', 'bed'], array(['Qty', 'Form', 'FOR', 'Duration', 'PeriodUnit', 'AT', 'WHEN'],
      dtype=object))


In [186]:
predictions = predict("5 days 2 tabs at bed")
print (predictions)

(['5', 'days', '2', 'tabs', 'at', 'bed'], array(['Duration', 'DurationUnit', 'Qty', 'Form', 'AT', 'WHEN'],
      dtype=object))


In [188]:
predictions = predict("3 tabs qid x 10 weeks")
print (predictions)

(['3', 'tabs', 'qid', 'x', '10', 'weeks'], array(['Qty', 'Form', 'Frequency', 'FOR', 'Duration', 'DurationUnit'],
      dtype=object))


In [190]:
predictions = predict("x 30 days")
print (predictions)

(['x', '30', 'days'], array(['FOR', 'Duration', 'DurationUnit'], dtype=object))


In [192]:
predictions = predict("x 20 months")
print (predictions)

(['x', '20', 'months'], array(['FOR', 'Duration', 'DurationUnit'], dtype=object))


In [194]:
predictions = predict("take 2 tabs po tid for 10 days")
print (predictions)

(['take', '2', 'tabs', 'po', 'tid', 'for', '10', 'days'], array(['Method', 'Qty', 'Form', 'PO', 'TID', 'FOR', 'Duration',
       'DurationUnit'], dtype=object))


In [196]:
predictions = predict("take 2 capsules po every 6 hours")
print (predictions)

(['take', '2', 'capsules', 'po', 'every', '6', 'hours'], array(['Method', 'Qty', 'Form', 'PO', 'EVERY', 'Period', 'PeriodUnit'],
      dtype=object))


In [198]:
predictions = predict("inject 2 units pu tid")
print (predictions)

(['inject', '2', 'units', 'pu', 'tid'], array(['Method', 'Qty', 'Form', 'Frequency', 'TID'], dtype=object))


In [200]:
predictions = predict("swallow 3 caps tid by mouth")
print (predictions)

(['swallow', '3', 'caps', 'tid', 'by', 'mouth'], array(['Method', 'Qty', 'Form', 'Frequency', 'Period', 'PeriodUnit'],
      dtype=object))


In [202]:
predictions = predict("inject 3 units orally")
print (predictions)

(['inject', '3', 'units', 'orally'], array(['Method', 'Qty', 'Form', 'Frequency'], dtype=object))


In [204]:
predictions = predict("orally take 3 tabs tid")
print (predictions)

(['orally', 'take', '3', 'tabs', 'tid'], array(['Method', 'Method', 'Qty', 'Form', 'TID'], dtype=object))


In [206]:
predictions = predict("by mouth take three caps")
print (predictions)

(['by', 'mouth', 'take', 'three', 'caps'], array(['Qty', 'Form', 'Method', 'Qty', 'Form'], dtype=object))


In [208]:
predictions = predict("take 3 tabs orally three times a day for 10 days at bedtime")
print (predictions)

(['take', '3', 'tabs', 'orally', 'three', 'times', 'a', 'day', 'for', '10', 'days', 'at', 'bedtime'], array(['Method', 'Qty', 'Form', 'Method', 'Qty', 'TIMES', 'Period',
       'PeriodUnit', 'FOR', 'Duration', 'DurationUnit', 'AT', 'WHEN'],
      dtype=object))


In [210]:
predictions = predict("take 3 tabs orally bid for 10 days at bedtime")
print (predictions)

(['take', '3', 'tabs', 'orally', 'bid', 'for', '10', 'days', 'at', 'bedtime'], array(['Method', 'Qty', 'Form', 'Frequency', 'QID', 'FOR', 'Duration',
       'DurationUnit', 'AT', 'WHEN'], dtype=object))


In [212]:
predictions = predict("take 3 tabs bid orally at bed")
print (predictions)

(['take', '3', 'tabs', 'bid', 'orally', 'at', 'bed'], array(['Method', 'Qty', 'Form', 'Frequency', 'DAILY', 'AT', 'WHEN'],
      dtype=object))


In [214]:
predictions = predict("take 10 capsules by mouth qid")
print (predictions)

(['take', '10', 'capsules', 'by', 'mouth', 'qid'], array(['Method', 'Qty', 'Form', 'BY', 'PO', 'QID'], dtype=object))


In [216]:
predictions = predict("inject 10 units orally qid x 3 months")
print (predictions)

(['inject', '10', 'units', 'orally', 'qid', 'x', '3', 'months'], array(['Method', 'Qty', 'Form', 'Frequency', 'QID', 'Method', 'Qty',
       'Form'], dtype=object))


In [220]:
prediction = predict("please take 2 tablets per day for a month in the morning and evening each day")
print (prediction)

(['please', 'take', '2', 'tablets', 'per', 'day', 'for', 'a', 'month', 'in', 'the', 'morning', 'and', 'evening', 'each', 'day'], array(['Method', 'Method', 'Qty', 'Form', 'Frequency', 'PeriodUnit',
       'FOR', 'Period', 'PeriodUnit', 'EVERY', 'Period', 'PeriodUnit',
       'AND', 'EVERY', 'Period', 'PeriodUnit'], dtype=object))


In [222]:
prediction = predict("Amoxcicillin QID 30 tablets")
print (prediction)

(['Amoxcicillin', 'QID', '30', 'tablets'], array(['Method', 'Method', 'Qty', 'Form'], dtype=object))


In [224]:
prediction = predict("take 3 tabs TID for 90 days with food")
print (prediction)

(['take', '3', 'tabs', 'TID', 'for', '90', 'days', 'with', 'food'], array(['Method', 'Qty', 'Form', 'Frequency', 'FOR', 'Duration',
       'DurationUnit', 'WITH', 'FOOD'], dtype=object))


In [226]:
prediction = predict("with food take 3 tablets per day for 90 days")
print (prediction)

(['with', 'food', 'take', '3', 'tablets', 'per', 'day', 'for', '90', 'days'], array(['WITH', 'FOOD', 'Method', 'Qty', 'Form', 'Frequency', 'PeriodUnit',
       'FOR', 'Duration', 'DurationUnit'], dtype=object))


In [228]:
prediction = predict("with food take 3 tablets per week for 90 weeks")
print (prediction)

(['with', 'food', 'take', '3', 'tablets', 'per', 'week', 'for', '90', 'weeks'], array(['WITH', 'FOOD', 'Method', 'Qty', 'Form', 'Frequency', 'PeriodUnit',
       'FOR', 'Duration', 'DurationUnit'], dtype=object))


In [230]:
prediction = predict("take 2-4 tabs")
print (prediction)

(['take', '2-4', 'tabs'], array(['Method', 'Qty', 'Form'], dtype=object))


In [232]:
prediction = predict("take 2 to 4 tabs")
print (prediction)

(['take', '2', 'to', '4', 'tabs'], array(['Method', 'Qty', 'TO', 'Qty', 'Form'], dtype=object))


In [234]:
prediction = predict("take two to four tabs")
print (prediction)

(['take', 'two', 'to', 'four', 'tabs'], array(['Method', 'Qty', 'TO', 'Qty', 'Form'], dtype=object))


In [236]:
prediction = predict("take 2-4 tabs for 8 to 9 days")
print (prediction)

(['take', '2-4', 'tabs', 'for', '8', 'to', '9', 'days'], array(['Method', 'Qty', 'Form', 'FOR', 'Duration', 'TO', 'PeriodMax',
       'PeriodUnit'], dtype=object))


In [238]:
prediction = predict("take 20 tabs every 6 to 8 days")
print (prediction)

(['take', '20', 'tabs', 'every', '6', 'to', '8', 'days'], array(['Method', 'Qty', 'Form', 'EVERY', 'Period', 'TO', 'PeriodMax',
       'PeriodUnit'], dtype=object))


In [240]:
prediction = predict("take 2 tabs every 4 to 6 days")
print (prediction)

(['take', '2', 'tabs', 'every', '4', 'to', '6', 'days'], array(['Method', 'Qty', 'Form', 'EVERY', 'Period', 'TO', 'PeriodMax',
       'PeriodUnit'], dtype=object))


In [242]:
prediction = predict("take 2 tabs every 2 to 10 weeks")
print (prediction)

(['take', '2', 'tabs', 'every', '2', 'to', '10', 'weeks'], array(['Method', 'Qty', 'Form', 'EVERY', 'Period', 'TO', 'Duration',
       'DurationUnit'], dtype=object))


In [244]:
prediction = predict("take 2 tabs every 4 to 6 days")
print (prediction)

(['take', '2', 'tabs', 'every', '4', 'to', '6', 'days'], array(['Method', 'Qty', 'Form', 'EVERY', 'Period', 'TO', 'PeriodMax',
       'PeriodUnit'], dtype=object))


In [248]:
prediction = predict("take 2 tabs every 2 to 10 months")
print (prediction)

(['take', '2', 'tabs', 'every', '2', 'to', '10', 'months'], array(['Method', 'Qty', 'Form', 'EVERY', 'Period', 'TO', 'PeriodMax',
       'PeriodUnit'], dtype=object))


In [250]:
prediction = predict("every 60 mins")
print (prediction)

(['every', '60', 'mins'], array(['EVERY', 'Period', 'PeriodUnit'], dtype=object))


In [252]:
prediction = predict("every 10 mins")
print (prediction)

(['every', '10', 'mins'], array(['EVERY', 'Period', 'PeriodUnit'], dtype=object))


In [254]:
prediction = predict("every two to four months")
print (prediction)

(['every', 'two', 'to', 'four', 'months'], array(['EVERY', 'Period', 'TO', 'PeriodMax', 'PeriodUnit'], dtype=object))


In [256]:
prediction = predict("take 2 tabs every 3 to 4 days")
print (prediction)

(['take', '2', 'tabs', 'every', '3', 'to', '4', 'days'], array(['Method', 'Qty', 'Form', 'EVERY', 'Period', 'TO', 'PeriodMax',
       'PeriodUnit'], dtype=object))


In [258]:
prediction = predict("every 3 to 4 days take 20 tabs")
print (prediction)

(['every', '3', 'to', '4', 'days', 'take', '20', 'tabs'], array(['EVERY', 'Period', 'TO', 'Qty', 'Form', 'Method', 'Qty', 'Form'],
      dtype=object))


In [260]:
prediction = predict("once in every 3 days take 3 tabs")
print (prediction)

(['once', 'in', 'every', '3', 'days', 'take', '3', 'tabs'], array(['Frequency', 'Period', 'EVERY', 'Period', 'PeriodUnit', 'Method',
       'Qty', 'Form'], dtype=object))


In [262]:
prediction = predict("take 3 tabs once in every 3 days")
print (prediction)

(['take', '3', 'tabs', 'once', 'in', 'every', '3', 'days'], array(['Method', 'Qty', 'Form', 'Frequency', 'Period', 'EVERY', 'Period',
       'PeriodUnit'], dtype=object))


In [264]:
prediction = predict("orally take 20 tabs every 4-6 weeks")
print (prediction)

(['orally', 'take', '20', 'tabs', 'every', '4-6', 'weeks'], array(['Method', 'Method', 'Qty', 'Form', 'EVERY', 'Period', 'PeriodUnit'],
      dtype=object))


In [266]:
prediction = predict("10 tabs x 2 days")
print (prediction)

(['10', 'tabs', 'x', '2', 'days'], array(['Qty', 'Form', 'FOR', 'Duration', 'DurationUnit'], dtype=object))


In [268]:
prediction = predict("3 capsule x 15 days")
print (prediction)

(['3', 'capsule', 'x', '15', 'days'], array(['Qty', 'Form', 'FOR', 'Duration', 'DurationUnit'], dtype=object))


In [270]:
prediction = predict("10 tabs")
print (prediction)

(['10', 'tabs'], array(['Qty', 'Form'], dtype=object))
